# MathScy Conjecture Generation & Evaluation Pipeline

This notebook implements:
1. **Domain-Conditioned Conjecture Generation** - Generate novel math conjectures per MoE expert domain
2. **Multi-Strategy Generation** - Pattern interpolation, analogy transfer, theorem composition, boundary exploration
3. **Self-Play Theorem Prover (STP)** - Conjecturer + prover mutual training loop
4. **4-Tier Evaluation Pipeline** - Type-check → triviality → counterexample → proof attempt
5. **Conjecture Ranking & Selection** - Score and rank generated conjectures for novelty and interest

In [ ]:
import os, sys, json, time, random
import numpy as np
from collections import defaultdict, Counter
from pathlib import Path
from typing import List, Dict, Optional, Tuple

sys.path.insert(0, '/scratch/ctoxtli/moexp/scripts')
from llm_utils import (
    gemini_generate, gemini_batch_generate,
    hpc_generate, hpc_chat_completion,
    CONJECTURE_GENERATION_PROMPT
)
from data_utils import get_domain_group, get_primary_category, MATH_DOMAIN_GROUPS

with open('/scratch/ctoxtli/moexp/configs/project_config.json') as f:
    config = json.load(f)

RESULTS_DIR = config['paths']['results_dir']
DATA_DIR = config['paths']['data_dir']
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print('Setup complete.')

## 1. Load Extracted Knowledge Base

Load theorems/results extracted in notebook 01 and organize by domain.

In [ ]:
def load_knowledge_base(extracted_path: str) -> Dict[str, List[Dict]]:
    """Load extracted knowledge and organize by MoE expert domain."""
    domain_knowledge = defaultdict(list)
    
    if not os.path.exists(extracted_path):
        print(f'Knowledge base not found at {extracted_path}')
        print('Run notebook 01 first to extract knowledge.')
        return domain_knowledge
    
    with open(extracted_path) as f:
        for line in f:
            entry = json.loads(line)
            domain = entry.get('domain_group', 'other')
            extracted = entry.get('extracted', {})
            if not extracted.get('parse_error'):
                domain_knowledge[domain].append({
                    'id': entry.get('id'),
                    'domain': domain,
                    'statement_type': extracted.get('statement_type', 'unknown'),
                    'formal_statement': extracted.get('formal_statement', ''),
                    'informal_description': extracted.get('informal_description', ''),
                    'key_concepts': extracted.get('key_concepts', []),
                    'potential_generalizations': extracted.get('potential_generalizations', ''),
                    'related_theorems': extracted.get('related_theorems', []),
                })
    
    print(f'Knowledge base loaded:')
    for domain, entries in sorted(domain_knowledge.items()):
        print(f'  {domain}: {len(entries)} entries')
    
    return domain_knowledge

extracted_path = os.path.join(DATA_DIR, 'extracted_knowledge.jsonl')
knowledge_base = load_knowledge_base(extracted_path)

## 2. Multi-Strategy Conjecture Generation

We implement 4 complementary strategies for generating novel conjectures:

1. **Pattern Interpolation** - Find patterns across theorems in the same domain and interpolate
2. **Cross-Domain Analogy** - Transfer structural patterns between related domains
3. **Theorem Composition** - Combine results to form new statements
4. **Boundary Exploration** - Push known results to their limits

In [ ]:
# Strategy 1: Pattern Interpolation Prompt
PATTERN_INTERPOLATION_PROMPT = """You are a mathematical researcher specializing in {domain}.

Analyze the following collection of theorems/results from {domain} and identify common patterns.
Then generate novel conjectures by interpolating between these patterns.

Known Results:
{results}

Instructions:
1. Identify structural patterns shared across these results
2. Find parameters or conditions that vary between results  
3. Interpolate: what happens for intermediate values or conditions?
4. Generate conjectures that fill "gaps" in the pattern

Return a JSON array of conjecture objects, each with:
{{
    "strategy": "pattern_interpolation",
    "conjecture_statement": "precise LaTeX statement",
    "informal_description": "plain English",
    "pattern_identified": "what pattern was found",
    "source_results": ["which results inspire this"],
    "confidence": 0.0-1.0,
    "estimated_difficulty": "easy|medium|hard|open_problem"
}}

Return ONLY valid JSON."""


# Strategy 2: Cross-Domain Analogy Prompt
CROSS_DOMAIN_PROMPT = """You are a creative mathematician who finds deep connections between different fields.

Source domain ({source_domain}) results:
{source_results}

Target domain ({target_domain}) context:
{target_results}

Instructions:
1. Identify structural similarities between source and target domains
2. Find theorems in the source domain that have no known analogue in the target
3. Formulate analogous conjectures for the target domain
4. Ensure the analogy is mathematically meaningful, not just superficial

Return a JSON array of conjecture objects, each with:
{{
    "strategy": "cross_domain_analogy",
    "conjecture_statement": "precise LaTeX statement in target domain",
    "informal_description": "plain English",
    "source_theorem": "the theorem being analogized",
    "analogy_mapping": "how concepts map between domains",
    "confidence": 0.0-1.0,
    "estimated_difficulty": "easy|medium|hard|open_problem"
}}

Return ONLY valid JSON."""


# Strategy 3: Theorem Composition Prompt  
COMPOSITION_PROMPT = """You are a mathematical researcher who combines existing results to discover new ones.

Given these results from {domain}:
{results}

Instructions:
1. Consider pairs or triples of results that share key concepts
2. Ask: what happens when we apply one result's technique to another's objects?
3. Look for results that, combined, might imply something new
4. Check if the converse of any combined result could hold

Return a JSON array of conjecture objects, each with:
{{
    "strategy": "theorem_composition",
    "conjecture_statement": "precise LaTeX statement",
    "informal_description": "plain English",
    "composed_from": ["result 1", "result 2"],
    "composition_method": "how the results were combined",
    "confidence": 0.0-1.0,
    "estimated_difficulty": "easy|medium|hard|open_problem"
}}

Return ONLY valid JSON."""


# Strategy 4: Boundary Exploration Prompt
BOUNDARY_PROMPT = """You are a mathematical researcher who probes the limits of known results.

Given these results from {domain}:
{results}

Instructions:
1. For each result, identify its hypotheses/conditions
2. Ask: what if we weaken a hypothesis? Strengthen a conclusion?
3. What happens at boundary cases (n→∞, ε→0, dimension changes)?
4. Are the conditions necessary? Can any be removed?
5. Generate conjectures about these boundary behaviors

Return a JSON array of conjecture objects, each with:
{{
    "strategy": "boundary_exploration",
    "conjecture_statement": "precise LaTeX statement",
    "informal_description": "plain English",
    "original_result": "the result being pushed",
    "boundary_type": "weakened_hypothesis|strengthened_conclusion|limit_case|necessity",
    "confidence": 0.0-1.0,
    "estimated_difficulty": "easy|medium|hard|open_problem"
}}

Return ONLY valid JSON."""

print('Conjecture generation prompts defined.')

In [ ]:
def format_results_for_prompt(entries: List[Dict], max_entries: int = 8) -> str:
    """Format knowledge base entries as text for LLM prompts."""
    selected = entries[:max_entries] if len(entries) > max_entries else entries
    parts = []
    for i, e in enumerate(selected, 1):
        stmt = e.get('formal_statement', e.get('informal_description', 'N/A'))
        concepts = ', '.join(e.get('key_concepts', [])[:5])
        parts.append(f"Result {i} ({e.get('statement_type', 'unknown')}):\n"
                     f"  Statement: {stmt[:300]}\n"
                     f"  Key concepts: {concepts}\n")
    return '\n'.join(parts)


def generate_conjectures_for_domain(
    domain: str,
    knowledge: List[Dict],
    strategies: List[str] = None,
    cross_domain_source: Optional[Tuple[str, List[Dict]]] = None,
    model: str = 'gemini-2.0-flash',
    gemini_key: str = None,
) -> List[Dict]:
    """Generate conjectures for a domain using multiple strategies."""
    if strategies is None:
        strategies = ['pattern_interpolation', 'composition', 'boundary_exploration']
    
    all_conjectures = []
    results_text = format_results_for_prompt(knowledge)
    
    for strategy in strategies:
        print(f"  Generating with strategy: {strategy}...")
        
        if strategy == 'pattern_interpolation':
            prompt = PATTERN_INTERPOLATION_PROMPT.format(
                domain=domain, results=results_text)
        elif strategy == 'cross_domain_analogy' and cross_domain_source:
            src_domain, src_knowledge = cross_domain_source
            src_text = format_results_for_prompt(src_knowledge)
            prompt = CROSS_DOMAIN_PROMPT.format(
                source_domain=src_domain, source_results=src_text,
                target_domain=domain, target_results=results_text)
        elif strategy == 'composition':
            prompt = COMPOSITION_PROMPT.format(
                domain=domain, results=results_text)
        elif strategy == 'boundary_exploration':
            prompt = BOUNDARY_PROMPT.format(
                domain=domain, results=results_text)
        else:
            continue
        
        response = gemini_generate(
            prompt, model=model, key=gemini_key,
            temperature=0.8, max_tokens=4096)
        
        # Parse response
        try:
            clean = response.strip()
            if clean.startswith('```'):
                clean = clean.split('\n', 1)[1]
            if clean.endswith('```'):
                clean = clean.rsplit('```', 1)[0]
            conjectures = json.loads(clean)
            if isinstance(conjectures, list):
                for c in conjectures:
                    c['domain'] = domain
                    c['strategy'] = strategy
                all_conjectures.extend(conjectures)
                print(f"    Generated {len(conjectures)} conjectures")
        except json.JSONDecodeError:
            print(f"    Failed to parse response for {strategy}")
        
        time.sleep(2)  # Rate limiting
    
    return all_conjectures

print('Conjecture generation functions defined.')

## 3. Generate Conjectures Across All Domains

In [ ]:
# Load Gemini key
with open('/scratch/ctoxtli/moexp/working_Gemini_API_keys.json') as f:
    gemini_keys = json.load(f)
working_key = gemini_keys[3]  # Key 4 (index 3) confirmed working

# Checkpoint path
conjectures_path = os.path.join(RESULTS_DIR, 'generated_conjectures.jsonl')

# Resume from checkpoint
processed_domains = set()
all_conjectures = []
if os.path.exists(conjectures_path):
    with open(conjectures_path) as f:
        for line in f:
            c = json.loads(line)
            all_conjectures.append(c)
            processed_domains.add(c.get('domain'))
    print(f'Resumed: {len(all_conjectures)} conjectures from {len(processed_domains)} domains')

# Priority domains (aligned with NSF proposal focus)
priority_domains = [
    'algebraic_geometry',  # Primary focus
    'combinatorics',       # Primary focus  
    'computational',       # Primary focus
    'algebra',
    'number_theory',
    'geometry_topology',
    'analysis',
]

# Cross-domain analogy pairs (structurally related domains)
analogy_pairs = [
    ('algebraic_geometry', 'number_theory'),
    ('combinatorics', 'algebra'),
    ('geometry_topology', 'algebraic_geometry'),
    ('analysis', 'probability_statistics'),
]

# Generate for each domain
for domain in priority_domains:
    if domain in processed_domains:
        print(f'Skipping {domain} (already processed)')
        continue
    
    if domain not in knowledge_base or len(knowledge_base[domain]) < 3:
        print(f'Skipping {domain} (insufficient knowledge base: {len(knowledge_base.get(domain, []))} entries)')
        continue
    
    print(f'\n=== Generating conjectures for {domain} ===')
    
    # Determine strategies
    strategies = ['pattern_interpolation', 'composition', 'boundary_exploration']
    cross_source = None
    
    # Add cross-domain if pair exists
    for src, tgt in analogy_pairs:
        if tgt == domain and src in knowledge_base and len(knowledge_base[src]) >= 3:
            strategies.append('cross_domain_analogy')
            cross_source = (src, knowledge_base[src])
            break
    
    domain_conjectures = generate_conjectures_for_domain(
        domain=domain,
        knowledge=knowledge_base[domain],
        strategies=strategies,
        cross_domain_source=cross_source,
        gemini_key=working_key,
    )
    
    # Save incrementally
    with open(conjectures_path, 'a') as f:
        for c in domain_conjectures:
            f.write(json.dumps(c) + '\n')
    
    all_conjectures.extend(domain_conjectures)
    processed_domains.add(domain)
    print(f'  Total: {len(domain_conjectures)} conjectures for {domain}')

print(f'\n=== Total conjectures generated: {len(all_conjectures)} ===')

## 4. Self-Play Theorem Prover (STP) Loop

Implements the STP approach where:
- **Conjecturer** generates candidate statements
- **Prover** attempts to prove/disprove them
- Both improve through the interaction

This section prepares the STP training data and loop for GPU execution.

In [ ]:
class SelfPlayTheoremProver:
    """Self-Play Theorem Prover (STP) - Conjecturer-Prover mutual training.
    
    Based on: Self-play theorem proving approaches where a conjecturer
    generates candidates and a prover attempts verification, with both
    models improving through the feedback loop.
    
    For GPU execution: The models are loaded and fine-tuned on GPU.
    For CPU: We use LLM APIs (Gemini/HPC) as stand-ins.
    """
    
    def __init__(self, use_gpu=False, gemini_key=None):
        self.use_gpu = use_gpu
        self.gemini_key = gemini_key
        self.conjecturer_model = None
        self.prover_model = None
        self.conjecture_history = []
        self.proof_history = []
        self.round_stats = []
        
    def init_gpu_models(self, conjecturer_path: str, prover_path: str):
        """Initialize GPU models for STP. Call on GPU machine."""
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        
        print('Loading conjecturer model...')
        self.conjecturer_tokenizer = AutoTokenizer.from_pretrained(conjecturer_path)
        self.conjecturer_model = AutoModelForCausalLM.from_pretrained(
            conjecturer_path, torch_dtype=torch.bfloat16, device_map='auto')
        
        print('Loading prover model...')
        self.prover_tokenizer = AutoTokenizer.from_pretrained(prover_path)
        self.prover_model = AutoModelForCausalLM.from_pretrained(
            prover_path, torch_dtype=torch.bfloat16, device_map='auto')
        
        print('STP models loaded.')
    
    def generate_conjecture_api(self, domain: str, context: str) -> Dict:
        """Generate a conjecture using LLM API (CPU fallback)."""
        prompt = f"""You are a creative mathematician in {domain}.
        
Given this mathematical context:
{context[:1500]}

Generate ONE novel mathematical conjecture that:
1. Is precise and falsifiable
2. Is non-trivial (not immediately obvious)
3. Could potentially be proven using known techniques
4. Can be formalized in Lean 4

Return JSON:
{{
    "conjecture": "precise LaTeX statement",
    "informal": "plain English",
    "lean4_sketch": "approximate Lean 4 code",
    "proof_hint": "suggested approach to prove or disprove"
}}

Return ONLY valid JSON."""
        
        response = gemini_generate(
            prompt, key=self.gemini_key, temperature=0.9, max_tokens=2048)
        
        try:
            clean = response.strip()
            if clean.startswith('```'):
                clean = clean.split('\n', 1)[1]
            if clean.endswith('```'):
                clean = clean.rsplit('```', 1)[0]
            return json.loads(clean)
        except:
            return {'conjecture': response, 'parse_error': True}
    
    def attempt_proof_api(self, conjecture: str, domain: str) -> Dict:
        """Attempt to prove/disprove a conjecture using LLM API."""
        prompt = f"""You are a mathematical proof assistant specializing in {domain}.
        
Analyze the following conjecture and attempt to either prove it, disprove it with a counterexample, or explain why it's difficult.

Conjecture: {conjecture}

Return JSON:
{{
    "verdict": "proved|disproved|unknown|trivial",
    "proof_or_counterexample": "the proof, counterexample, or analysis",
    "lean4_proof_sketch": "approximate Lean 4 proof if verdict is proved",
    "difficulty_assessment": "easy|medium|hard|very_hard|open_problem",
    "confidence": 0.0-1.0,
    "feedback_for_conjecturer": "what makes this a good/bad conjecture"
}}

Return ONLY valid JSON."""
        
        response = gemini_generate(
            prompt, key=self.gemini_key, temperature=0.3, max_tokens=4096)
        
        try:
            clean = response.strip()
            if clean.startswith('```'):
                clean = clean.split('\n', 1)[1]
            if clean.endswith('```'):
                clean = clean.rsplit('```', 1)[0]
            return json.loads(clean)
        except:
            return {'verdict': 'unknown', 'raw_response': response}
    
    def run_stp_round(self, domain: str, knowledge: List[Dict], n_conjectures: int = 5) -> Dict:
        """Run one round of self-play: generate conjectures, attempt proofs."""
        round_results = {
            'domain': domain,
            'conjectures': [],
            'stats': {'total': 0, 'proved': 0, 'disproved': 0, 'unknown': 0, 'trivial': 0}
        }
        
        # Create context from knowledge base
        context = format_results_for_prompt(knowledge, max_entries=5)
        
        for i in range(n_conjectures):
            print(f"    Conjecture {i+1}/{n_conjectures}...")
            
            # Generate
            conjecture = self.generate_conjecture_api(domain, context)
            if conjecture.get('parse_error'):
                continue
            
            # Attempt proof
            conj_text = conjecture.get('conjecture', '')
            proof_result = self.attempt_proof_api(conj_text, domain)
            
            # Record
            result = {
                'conjecture': conjecture,
                'proof_attempt': proof_result,
                'domain': domain,
            }
            round_results['conjectures'].append(result)
            
            verdict = proof_result.get('verdict', 'unknown')
            round_results['stats']['total'] += 1
            round_results['stats'][verdict] = round_results['stats'].get(verdict, 0) + 1
            
            self.conjecture_history.append(result)
            
            time.sleep(3)  # Rate limiting
        
        self.round_stats.append(round_results['stats'])
        return round_results
    
    def run_stp_loop(
        self, domain: str, knowledge: List[Dict],
        n_rounds: int = 3, n_per_round: int = 5,
        checkpoint_path: str = None,
    ) -> List[Dict]:
        """Run multiple rounds of STP."""
        all_round_results = []
        
        # Resume from checkpoint
        start_round = 0
        if checkpoint_path and os.path.exists(checkpoint_path):
            with open(checkpoint_path) as f:
                checkpoint = json.load(f)
            all_round_results = checkpoint.get('rounds', [])
            start_round = len(all_round_results)
            self.conjecture_history = checkpoint.get('all_conjectures', [])
            print(f'Resuming STP from round {start_round}')
        
        for round_num in range(start_round, n_rounds):
            print(f'\n  === STP Round {round_num + 1}/{n_rounds} for {domain} ===')
            
            round_results = self.run_stp_round(domain, knowledge, n_per_round)
            all_round_results.append(round_results)
            
            # Save checkpoint
            if checkpoint_path:
                checkpoint_data = {
                    'domain': domain,
                    'rounds': all_round_results,
                    'all_conjectures': self.conjecture_history,
                    'round_stats': self.round_stats,
                }
                with open(checkpoint_path, 'w') as f:
                    json.dump(checkpoint_data, f, indent=2)
            
            stats = round_results['stats']
            print(f'  Round {round_num + 1} stats: {stats}')
        
        return all_round_results

print('SelfPlayTheoremProver class defined.')

In [ ]:
# Run STP for priority domains
stp = SelfPlayTheoremProver(gemini_key=working_key)

stp_results = {}
for domain in ['algebraic_geometry', 'combinatorics', 'computational']:
    if domain not in knowledge_base or len(knowledge_base[domain]) < 3:
        print(f'Skipping STP for {domain} (insufficient data)')
        continue
    
    checkpoint = os.path.join(RESULTS_DIR, f'stp_{domain}_checkpoint.json')
    print(f'\n{"="*60}')
    print(f'Running STP for {domain}')
    
    results = stp.run_stp_loop(
        domain=domain,
        knowledge=knowledge_base[domain],
        n_rounds=3,
        n_per_round=5,
        checkpoint_path=checkpoint,
    )
    stp_results[domain] = results

print('\n=== STP Complete ===')
print(f'Total conjectures explored: {len(stp.conjecture_history)}')

## 5. GPU-Ready STP Training Data Preparation

Prepare paired (conjecture, proof_attempt) data for fine-tuning the MoE model's conjecturer and prover heads.

In [ ]:
def prepare_stp_training_data(conjecture_history: List[Dict]) -> Dict[str, List[Dict]]:
    """Prepare training data from STP results for fine-tuning."""
    conjecturer_data = []  # For training the conjecturer
    prover_data = []       # For training the prover
    
    for entry in conjecture_history:
        conjecture = entry.get('conjecture', {})
        proof = entry.get('proof_attempt', {})
        domain = entry.get('domain', 'unknown')
        verdict = proof.get('verdict', 'unknown')
        
        conj_text = conjecture.get('conjecture', '')
        if not conj_text:
            continue
        
        # Conjecturer training: reward non-trivial, provable conjectures
        quality_score = 0.0
        if verdict == 'proved':
            quality_score = 1.0
        elif verdict == 'unknown':
            quality_score = 0.7  # Hard conjectures are interesting
        elif verdict == 'disproved':
            quality_score = 0.3  # Some signal, but wrong
        elif verdict == 'trivial':
            quality_score = 0.0  # Not useful
        
        conjecturer_data.append({
            'domain': domain,
            'instruction': f'Generate a novel mathematical conjecture in {domain}.',
            'response': conj_text,
            'quality_score': quality_score,
            'lean4_sketch': conjecture.get('lean4_sketch', ''),
        })
        
        # Prover training: learn to prove/disprove
        proof_text = proof.get('proof_or_counterexample', '')
        if proof_text:
            prover_data.append({
                'domain': domain,
                'instruction': f'Prove or disprove the following conjecture in {domain}:\n{conj_text}',
                'response': proof_text,
                'verdict': verdict,
                'lean4_proof': proof.get('lean4_proof_sketch', ''),
            })
    
    return {
        'conjecturer': conjecturer_data,
        'prover': prover_data,
    }

# Prepare and save training data
stp_training = prepare_stp_training_data(stp.conjecture_history)

for role in ['conjecturer', 'prover']:
    path = os.path.join(DATA_DIR, f'stp_{role}_training.jsonl')
    with open(path, 'w') as f:
        for entry in stp_training[role]:
            f.write(json.dumps(entry) + '\n')
    print(f'Saved {len(stp_training[role])} {role} training examples to {path}')

print('\nSTP training data prepared for GPU fine-tuning.')

## 6. Conjecture Ranking & Quality Assessment

In [ ]:
def score_conjecture(conjecture: Dict, proof_attempt: Dict) -> float:
    """Score a conjecture based on novelty, difficulty, and interest.
    
    Scoring criteria:
    - Novelty: Not trivially provable or disprovable (0.3 weight)
    - Difficulty: Harder conjectures are more interesting (0.2 weight)
    - Formalizability: Can it be stated in Lean 4? (0.2 weight)
    - Confidence: How confident is the prover? (0.15 weight)
    - Parsability: Well-formed mathematical statement (0.15 weight)
    """
    score = 0.0
    
    verdict = proof_attempt.get('verdict', 'unknown')
    
    # Novelty score
    novelty_map = {'trivial': 0.0, 'proved': 0.6, 'disproved': 0.3, 'unknown': 1.0}
    score += 0.3 * novelty_map.get(verdict, 0.5)
    
    # Difficulty score
    diff_map = {'easy': 0.2, 'medium': 0.5, 'hard': 0.8, 'very_hard': 0.95, 'open_problem': 1.0}
    diff = proof_attempt.get('difficulty_assessment', 'medium')
    score += 0.2 * diff_map.get(diff, 0.5)
    
    # Formalizability score
    lean_sketch = conjecture.get('lean4_sketch', '')
    if lean_sketch and len(lean_sketch) > 20 and 'theorem' in lean_sketch.lower():
        score += 0.2 * 0.9
    elif lean_sketch and len(lean_sketch) > 10:
        score += 0.2 * 0.5
    else:
        score += 0.2 * 0.1
    
    # Confidence
    confidence = proof_attempt.get('confidence', 0.5)
    if isinstance(confidence, (int, float)):
        score += 0.15 * (1.0 - abs(confidence - 0.5) * 2)  # Mid-confidence = interesting
    
    # Parsability
    conj_text = conjecture.get('conjecture', '') or conjecture.get('conjecture_statement', '')
    if conj_text and not conjecture.get('parse_error'):
        score += 0.15 * 0.8
    
    return round(score, 4)


def rank_conjectures(conjecture_history: List[Dict]) -> List[Dict]:
    """Rank all generated conjectures by quality score."""
    scored = []
    for entry in conjecture_history:
        conjecture = entry.get('conjecture', {})
        proof = entry.get('proof_attempt', {})
        score = score_conjecture(conjecture, proof)
        scored.append({
            **entry,
            'quality_score': score,
        })
    
    scored.sort(key=lambda x: -x['quality_score'])
    return scored


# Rank all conjectures
if stp.conjecture_history:
    ranked = rank_conjectures(stp.conjecture_history)
    
    print(f'Top 10 conjectures by quality score:')
    for i, entry in enumerate(ranked[:10], 1):
        conj = entry.get('conjecture', {})
        proof = entry.get('proof_attempt', {})
        print(f'\n{i}. Score: {entry["quality_score"]:.3f} | Domain: {entry.get("domain", "?")} | Verdict: {proof.get("verdict", "?")}')
        print(f'   {(conj.get("conjecture", "") or conj.get("conjecture_statement", ""))[:150]}')
    
    # Save ranked conjectures
    ranked_path = os.path.join(RESULTS_DIR, 'ranked_conjectures.json')
    with open(ranked_path, 'w') as f:
        json.dump(ranked, f, indent=2)
    print(f'\nSaved {len(ranked)} ranked conjectures to {ranked_path}')
else:
    print('No conjectures to rank yet. Run STP loop first.')

## 7. Lean 4 Verification Queue

Prepare top-ranked conjectures for Lean 4 verification (executed in notebook 03).

In [ ]:
def prepare_lean_verification_queue(
    ranked_conjectures: List[Dict],
    top_n: int = 50,
    output_path: str = None,
) -> List[Dict]:
    """Prepare top conjectures for Lean 4 verification."""
    queue = []
    
    for entry in ranked_conjectures[:top_n]:
        conjecture = entry.get('conjecture', {})
        lean_sketch = conjecture.get('lean4_sketch', '')
        conj_text = conjecture.get('conjecture', '') or conjecture.get('conjecture_statement', '')
        
        queue.append({
            'domain': entry.get('domain', 'unknown'),
            'quality_score': entry.get('quality_score', 0),
            'conjecture_latex': conj_text,
            'informal_description': conjecture.get('informal', '') or conjecture.get('informal_description', ''),
            'lean4_sketch': lean_sketch,
            'proof_hint': conjecture.get('proof_hint', ''),
            'prover_verdict': entry.get('proof_attempt', {}).get('verdict', 'unknown'),
            'verification_status': 'pending',
        })
    
    if output_path:
        with open(output_path, 'w') as f:
            json.dump(queue, f, indent=2)
        print(f'Saved {len(queue)} conjectures to Lean 4 verification queue: {output_path}')
    
    return queue

if stp.conjecture_history:
    ranked = rank_conjectures(stp.conjecture_history)
    lean_queue = prepare_lean_verification_queue(
        ranked,
        top_n=50,
        output_path=os.path.join(RESULTS_DIR, 'lean_verification_queue.json'),
    )
    print(f'\nConjectures queued for Lean 4 verification: {len(lean_queue)}')
    print('Run notebook 03 to verify these conjectures.')
else:
    print('No conjectures available yet.')

## 8. GPU Training Setup for STP Fine-tuning

Prepare configuration for fine-tuning MoE model on STP data (for GPU execution).

In [ ]:
stp_training_config = {
    'description': 'STP fine-tuning configuration for MathScy MoE',
    'base_model': 'deepseek-ai/deepseek-math-7b-instruct',
    'conjecturer': {
        'data_path': os.path.join(DATA_DIR, 'stp_conjecturer_training.jsonl'),
        'lora_config': {
            'r': 64,
            'lora_alpha': 128,
            'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
            'lora_dropout': 0.05,
        },
        'training': {
            'num_epochs': 3,
            'learning_rate': 2e-4,
            'batch_size': 4,
            'gradient_accumulation_steps': 8,
            'warmup_ratio': 0.05,
            'max_seq_length': 4096,
            'reward_model': 'quality_score',  # Use quality scores for RLHF
        },
    },
    'prover': {
        'base_model': 'deepseek-ai/DeepSeek-Prover-V2-7B',
        'data_path': os.path.join(DATA_DIR, 'stp_prover_training.jsonl'),
        'lora_config': {
            'r': 64,
            'lora_alpha': 128,
            'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
            'lora_dropout': 0.05,
        },
        'training': {
            'num_epochs': 3,
            'learning_rate': 1e-4,
            'batch_size': 4,
            'gradient_accumulation_steps': 8,
            'warmup_ratio': 0.05,
            'max_seq_length': 4096,
        },
    },
    'stp_loop': {
        'n_rounds': 10,
        'n_conjectures_per_round': 20,
        'retrain_interval': 3,  # Retrain models every 3 rounds
        'domains': ['algebraic_geometry', 'combinatorics', 'computational'],
    },
}

config_path = os.path.join('/scratch/ctoxtli/moexp/configs', 'stp_training_config.json')
with open(config_path, 'w') as f:
    json.dump(stp_training_config, f, indent=2)
print(f'STP training config saved to {config_path}')
print('\nTo run on GPU:')
print('1. Transfer data and config to DGX machine')
print('2. Run notebooks 02 (MoE training) then 04 (STP loop) with GPU')
print('3. Use notebook 03 for Lean 4 verification of results')

## 9. Summary & Pipeline Status

In [ ]:
# Summarize all results
summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'notebook': '04_conjecture_generation',
    'total_conjectures_generated': len(all_conjectures) + len(stp.conjecture_history),
    'multi_strategy_conjectures': len(all_conjectures),
    'stp_conjectures': len(stp.conjecture_history),
    'domains_covered': list(processed_domains),
    'stp_round_stats': stp.round_stats,
    'files_generated': {
        'conjectures': conjectures_path,
        'stp_training_conjecturer': os.path.join(DATA_DIR, 'stp_conjecturer_training.jsonl'),
        'stp_training_prover': os.path.join(DATA_DIR, 'stp_prover_training.jsonl'),
        'stp_config': config_path,
    },
    'next_steps': [
        'Run Lean 4 verification on top conjectures (notebook 03)',
        'GPU fine-tuning with STP data (notebook 02)',
        'Iterative STP loop on GPU for larger-scale conjecture generation',
    ],
}

summary_path = os.path.join(RESULTS_DIR, 'conjecture_generation_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('=== Conjecture Generation Pipeline Summary ===')
print(json.dumps(summary, indent=2))